In [0]:
import pyspark.sql.functions as F
# from pyspark.sql.types import IntegerType
df = spark.read.table("bronze_workforce.azure_blob_storage.general_ledgers")
display(df)

In [0]:
df=df.drop('_file','_line','_modified','_fivetran_synced')

In [0]:
df = df.withColumn("gl_id",df["gl_id"].cast('int')).withColumn("company_id",df["company_id"].cast('int')).withColumn("account_id",df["account_id"].cast('int')).withColumn("department_id",df["department_id"].cast('int'))
display(df)

In [0]:
df = df.withColumn("fiscal_year",df["fiscal_year"].cast('int')).withColumn("fiscal_period",df["fiscal_period"].cast('int')).withColumn("created_by",df["created_by"].cast('int')).withColumn("approved_by",df["approved_by"].cast('int'))
display(df)

In [0]:
df = df.withColumn('entry_date',F.trim(F.col("entry_date")))
df = df.withColumn("entry_date", F.to_timestamp(F.col("entry_date"), "dd-MM-yyyy HH:mm"))
df = df.withColumn("entry_date", F.to_date(F.col("entry_date"), "dd-MM-yyyy"))
display(df)

In [0]:
df = df.withColumn('posting_date',F.trim(F.col("posting_date")))
df = df.withColumn("posting_date", F.to_timestamp(F.col("posting_date"), "dd-MM-yyyy HH:mm"))
df = df.withColumn("posting_date", F.to_date(F.col("posting_date"), "dd-MM-yyyy"))
display(df)

In [0]:
df = df.dropDuplicates()
display(df)

In [0]:
unique_values = df.select("transaction_type").distinct()

# Display all unique values without truncation
unique_values.show(truncate=False)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("silver_workforce.azure_blob_storage.general_ledgers")